# 🩺 RAGnosis — BERT Medical Report Fine-Tuning
## PICT InC 2025

This notebook fine-tunes a BERT model on medical text classification.

**Task**: Classify medical report text into categories:
- Blood / CBC Report
- Diabetes / Glucose Report
- Lipid Panel Report
- Kidney Function Report
- Liver Function Report
- Thyroid Report
- General Medical Report

**Model**: `dmis-lab/biobert-base-cased-v1.1` (BERT fine-tuned on PubMed)

**Runtime**: Use GPU (T4) for faster training — Runtime → Change runtime type → T4 GPU

In [ ]:
# Install libraries
!pip install transformers datasets torch scikit-learn pandas numpy -q

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {device}')

In [ ]:
# ══════════════════════════════════
# SYNTHETIC TRAINING DATA
# ══════════════════════════════════
# In production, replace with real medical report text dataset

LABEL_NAMES = [
    'Blood / CBC Report', 'Diabetes / Glucose Report',
    'Lipid Panel Report', 'Kidney Function Report',
    'Liver Function Report', 'Thyroid Report', 'General Medical Report'
]
LABEL2ID = {l: i for i, l in enumerate(LABEL_NAMES)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

training_samples = [
    # Blood / CBC
    ("Patient hemoglobin 11.2 g/dL low anemia WBC 6500 RBC 4.1 platelets 220000", 0),
    ("Complete blood count CBC: HGB 13.5 WBC 8.2 thousand cells platelet count 180", 0),
    ("Red blood cells 4.8 white cells 7.1 hematocrit 42% MCV 88 fL", 0),
    ("Hemoglobin 15.0 g/dL WBC 11.5 high neutrophils elevated", 0),
    ("Blood analysis: PLT 95000 low thrombocytopenia RBC 3.9 mild anemia", 0),
    # Diabetes
    ("Fasting blood glucose 126 mg/dL HbA1c 7.2% diabetes mellitus type 2", 1),
    ("Random blood sugar 210 mg/dL postprandial glucose 185 insulin resistance", 1),
    ("FBS 98 mg/dL normal HbA1c 5.5% good glycemic control", 1),
    ("Glucose tolerance test OGTT 2hr value 160 mg/dL prediabetes", 1),
    ("Blood sugar fasting 145 glycated hemoglobin 8.1 uncontrolled diabetes", 1),
    # Lipid Panel
    ("Total cholesterol 245 mg/dL LDL 168 HDL 38 triglycerides 189 high", 2),
    ("Lipid profile: TC 198 LDL-C 120 HDL-C 55 TG 145 normal", 2),
    ("Dyslipidemia cholesterol 260 mg/dL very high cardiovascular risk", 2),
    ("Serum triglycerides 320 mg/dL elevated VLDL 64 HDL low 32", 2),
    ("Lipid panel fasting: total cholesterol 180 LDL 100 HDL 62 TG 90", 2),
    # Kidney
    ("Serum creatinine 2.1 mg/dL elevated BUN 32 mg/dL eGFR 38 CKD stage 3", 3),
    ("Kidney function test: creatinine 1.0 urea 18 normal renal function", 3),
    ("Urine protein 2+ microscopic hematuria creatinine 3.4 renal failure", 3),
    ("Renal panel: BUN 45 creatinine 2.8 GFR 22 severe kidney disease", 3),
    ("Creatinine clearance 65 mL/min mild kidney impairment BUN 24", 3),
    # Liver
    ("SGPT ALT 89 U/L elevated SGOT AST 67 bilirubin 2.1 mild hepatitis", 4),
    ("Liver function test LFT normal SGPT 32 SGOT 28 alkaline phosphatase 95", 4),
    ("Total bilirubin 4.5 direct 3.2 jaundice obstructive ALP 320 elevated", 4),
    ("Hepatic enzymes elevated ALT 450 AST 380 acute hepatocellular damage", 4),
    ("Liver cirrhosis serum albumin 2.8 prothrombin time prolonged", 4),
    # Thyroid
    ("TSH 8.5 mIU/L elevated hypothyroidism T4 free 0.6 low levothyroxine", 5),
    ("Thyroid function TFT: TSH 0.1 low hyperthyroid FT4 2.8 elevated", 5),
    ("TSH 2.1 normal FT3 3.2 FT4 1.4 euthyroid thyroid normal", 5),
    ("Anti-TPO antibody positive Hashimoto thyroiditis TSH 12.3 high", 5),
    ("Thyroid profile stimulating hormone 0.01 suppressed T3 toxicosis", 5),
    # General
    ("Routine health checkup all parameters within normal limits", 6),
    ("Vitamin D deficiency 14 ng/mL B12 180 pg/mL low supplement recommended", 6),
    ("Uric acid 8.2 mg/dL elevated gout risk purine diet restriction", 6),
    ("Chest X-ray normal cardiac shadow within limits no infiltrates", 6),
    ("Annual physical examination ESR 28 CRP 0.8 inflammatory markers normal", 6),
]

texts, labels = zip(*training_samples)
df = pd.DataFrame({'text': texts, 'label': labels})
print(f'✅ Dataset: {len(df)} samples, {len(LABEL_NAMES)} classes')
print(df['label'].value_counts())

In [ ]:
# ══════════════════════════════════
# TOKENIZER + MODEL
# ══════════════════════════════════
MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ══════════════════════════════════
# DATASET PREPARATION
# ══════════════════════════════════
from torch.utils.data import Dataset

class MedicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
train_dataset = MedicalDataset(train_texts, train_labels, tokenizer)
val_dataset = MedicalDataset(val_texts, val_labels, tokenizer)
print(f'✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
# ══════════════════════════════════
# TRAINING
# ══════════════════════════════════
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='./biobert_medical_classifier',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

In [ ]:
# ══════════════════════════════════
# EVALUATION
# ══════════════════════════════════
predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = [val_labels[i] for i in range(len(val_labels))]

print('\n📊 Classification Report:')
print(classification_report(true_labels, pred_labels, target_names=LABEL_NAMES))

accuracy = accuracy_score(true_labels, pred_labels)
print(f'\n✅ Final Accuracy: {accuracy * 100:.1f}%')

In [ ]:
# ══════════════════════════════════
# INFERENCE DEMO
# ══════════════════════════════════
model.eval()

test_texts = [
    "TSH 9.2 mIU/L, FT4 0.5, hypothyroidism suspected",
    "Hemoglobin 9.8 g/dL, RBC 3.8, MCV 72, iron deficiency anemia",
    "Total cholesterol 258 mg/dL, LDL 180, TG 220 mg/dL — elevated",
    "Blood glucose fasting 138 mg/dL, HbA1c 6.8% — diabetes",
    "Creatinine 1.9, BUN 38, eGFR 45 — Stage 3 CKD"
]

print('🔬 Inference Results:\n')
for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    pred_id = outputs.logits.argmax(-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1).max().item()
    print(f'  Input: {text[:60]}...')
    print(f'  → Predicted: {ID2LABEL[pred_id]} ({confidence*100:.1f}% confidence)')
    print()

In [ ]:
# ══════════════════════════════════
# SAVE MODEL
# ══════════════════════════════════
save_path = './biobert_ragnosis_final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f'✅ Model saved to: {save_path}')
print('\nTo use this model in RAGnosis backend:')
print('  1. Copy the ./biobert_ragnosis_final folder to backend/models/')
print('  2. Update backend/services/bert_summarizer.py MODEL_NAME to use local path')